# Random forest training for insurance claim amout
In this notebook, we use scikit-learn to train a Random Forest model that predicts the claim amount a customer is likely to request from the insurance company after an accident, based on their profile.

The model is trained directly on the raw data in csv format and not on the SQL data. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, mean_squared_log_error

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer

In [ ]:
# 1. Load dataset
data = pd.read_csv('trainset.csv')  # Update this if needed
data.head()

**dato che non posso più submittare le cose, faccio qui train test split** 

In [ ]:
# drop the ID
data.drop(columns="ID",inplace=True)

In [ ]:
X = data.drop(columns="AMT_Claim")
y = data["AMT_Claim"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

## Cose che vorrei provare

1. Istogramma di ogni feature (plot mean and variance) -> if low variance drop the features... or just do PCA
2. Correlazioni di ogni feature con ogni altra
3. Binning continuous variables

4. What does random predictions give you? What does constant predictions give you?
5. Using transformers to create meaningful encodings.

#### Encode categorical variables

In [ ]:
categorical_features = []
for key in X_train.keys():
    feature = X_train[key].iloc[0]
    if type(feature) == str:
        print(f"{key} unique values:",np.unique(X_train[key]))
        categorical_features.append((key,np.unique(X_train[key])))
        print("\n")
        

In [ ]:
X_train["Sex"] = X_train["Insured_Sex"].map({"Female":1, "Male":0})
X_train["Status"] = X_train["Insured_Status"].map({"Married":1, "Single": 0})
X_train["Area"] = X_train["Insured_DriveArea"].map({"Rural":1, "Urban": 0})
X_train["Car"] = X_train["Car_Use"].map({"Commercial": 0,
                                         "Commute":1,
                                         "Farmer":2,
                                         "Private":3})

In [ ]:
# # female is 1, male is 0 
# X_train["Sex"] = [1 if sex=="Female" else 0 for sex in X_train["Insured_Sex"] ]
# X_train["Status"] =  [1 if status=="Married" else 0 for status in X_train["Insured_Status"] ]
# X_train["Area"] =  [1 if area=="Rural" else 0 for area in X_train["Insured_DriveArea"] ]
# X_train["Car"] = [
#     0 if car == categorical_features[-1][1][0] 
#     else 1 if car == categorical_features[-1][1][1] 
#     else 2 if car == categorical_features[-1][1][2] 
#     else 3
#     for car in X_train["Car_Use"]
# ]


In [ ]:
# I can drop the old columns
X_train = X_train.drop(columns=["Insured_Sex","Insured_Status","Insured_DriveArea", "Car_Use"])

In [ ]:
X_train.head()

### Histograms

In [ ]:
X_train.hist(figsize=(20, 15), bins=50)

In [ ]:
y_train.hist(bins=50)
plt.yscale('log')

Potrei scegliere a bunch of models e assessare le performance per ogni azione che svolgo sul dataset.

* Scelgo solo random forest perché altrimenti ho troppe robe da provare.

I test da svolgere sono: (sempre in cross validation)

- Mutual information

1. Random prediction (predicting random values)
2. Constant prediction
3. Performance of model without doing nothing

5. Applying log to ouptuts
6. Normalization of features
7. Binning
8. PCA (standardized, continuos variables, use it for feature selection maybe)
9. K-Means Clustering to add a new feature

## Study feature importance with mutual information

In [ ]:
sns.catplot(pd.concat([X_train,y_train],axis=1),x="Car",y="AMT_Claim")
# "Commercial": 0,
# "Commute":1,
# "Farmer":2,
#  "Private":3})

In [ ]:
mir = mutual_info_regression(X_train,y_train)

In [ ]:
mir_scores = pd.Series(mir,name="MIR Scores",index=X_train.columns).sort_values(ascending=False)


In [ ]:
# Sort mutual information scores (descending order)
mir_scores_sorted = mir_scores.sort_values(ascending=True)  # ascending=True for barh (bottom=lowest, top=highest)

# Plot
plt.figure(figsize=(8, 12))  # width x height

width = np.arange(len(mir_scores_sorted))
ticks = list(mir_scores_sorted.index)

plt.barh(width, mir_scores_sorted)
plt.yticks(width, ticks)
plt.title("Mutual Information Scores")

plt.tight_layout()
plt.show()


## Let's start with the baselines

### Random prediction model
Doing no data transformation whatsover (keeping y not log-transformed)

In [ ]:
np.min(y_train), np.max(y_train)

In [ ]:
def my_random_model(size=1):
    return np.random.random(size)*np.max(y_train)

In [ ]:
mean_squared_log_error(y_train,my_random_model(len(y_train)))

In [ ]:
error = []
for maximum_value in np.linspace(0,np.max(y_train),100):
    error.append(mean_squared_log_error(y_train,np.random.random(len(y_train))*maximum_value))

In [ ]:
plt.plot(np.linspace(0,np.max(y_train),100),error)
plt.xlabel("Maximum possible value to be randomly sampled")
plt.ylabel("MSLE")

Let's try to zoom in.

I should take averages for each maximum value. This is a function of a random variable. You are interested in the expectations of MSLE fixing the maximum value. 

In [ ]:
rand_error_zoom     = []
rand_error_zoom_std = []

for maximum_value in np.linspace(0,1.5,100):
    # iterate 100 times and take the average and std
    running_msle = []
    for _ in range(1000):
        running_msle.append(mean_squared_log_error(y_train,np.random.random(len(y_train))*maximum_value))
    
    rand_error_zoom.append(np.mean(running_msle))
    rand_error_zoom_std.append(np.std(running_msle))
    
    #rand_error_zoom.append(mean_squared_log_error(y_train,np.random.random(len(y_train))*maximum_value))


In [ ]:
x = np.linspace(0, 1, 100)
y = np.array(rand_error_zoom)
yerr = np.array(rand_error_zoom_std)

plt.plot(x, y, label='Mean MSLE')
plt.fill_between(x, y - yerr, y + yerr, alpha=0.3, label='±1 std dev')
plt.xlabel("Zoom - Maximum possible value to be randomly sampled")
plt.ylabel("MSLE")
plt.legend()


What is it that we would like to minimize in this case? The mean value? The variance? Is there a formula to compute these?

### Constant prediction model

In [ ]:
error_const = []
for constant_value in np.linspace(0,np.max(y_train),1000):
    error_const.append(mean_squared_log_error(y_train,np.array([constant_value for _ in y_train])))

In [ ]:
plt.plot(error_const)

In [ ]:
closer_error = [mean_squared_log_error(y_train,np.array([constant_value for _ in y_train])) for constant_value in np.linspace(0,1,100)]

In [ ]:
from scipy.optimize import minimize

In [ ]:
def objective(y):
    return mean_squared_log_error(y_train,np.array([y for _ in range(len(y_train))]))

In [ ]:
minimize(objective,x0=.5)

In [ ]:
constant_values = np.linspace(0,1,100)
plt.plot(constant_values,closer_error)
plt.title("Performance predicting a constant value")
plt.plot(4.085e-01,2.677282138996734,"o")

plt.text(.2,2.75,f"Min error obtained with \nconstant pred y=4.099e-01\n loss=2.687")

I find that a naive model, predicting a constant value $y_{pred} = ~0.4$ achieves a loss of $l = ~2.687$.

This is the best naive baseline we want to defeat, using data science.

## Performance of model without doing nothing

In [ ]:
rf = RandomForestRegressor(verbose=10,n_jobs=-1)
rf

In [ ]:
scores = cross_val_score(rf, X_train, y_train, cv=5, n_jobs=-1,verbose=4,scoring=make_scorer(mean_squared_log_error))

In [ ]:
print("%0.2f accuracy with a standard deviation of %0.2f" % (scores.mean(), scores.std()))

This is NOT a trained model. It's needed for model selection (I think). Then, one should fit the actual random forest to have a trained model.

Let's try other models.

- GradientBoosting cannot be used together with MSLE for it produces negative outputs. I'd need to pre-process the data, which is something I'm planning to do in the next steps. 
- SVR is normalization dependent. This also implies that I will finetune it later (possibily with NNs too)


In [ ]:
models = {
    #'RandomForest': RandomForestRegressor(),
    #'GradientBoosting': GradientBoostingRegressor(),
    'SVR': SVR()
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train,
                             scoring=make_scorer(mean_squared_log_error),
                             cv=5, n_jobs=-1, verbose=0)
    print(f"{name} - RMSLE: {scores.mean():.4f} ± {scores.std():.4f}")


I found that RandomForest is the easiest model to train and it gives very good performance compared to the naive baseline. Let's see the performance on the whole train set.

In [ ]:
rf.fit(X_train,y_train)

In [ ]:
y_pred = rf.predict(X_train)

In [ ]:
mean_squared_log_error(y_train,y_pred)

The performance of the RandomForest on the train set is 0.24 vs. 2.68 of the constant model.

Mi mancano:

5. Applying log to ouptuts
6. Normalization of features -> GradientBoosting, SVM, NN
7. Binning
8. PCA (standardized, continuos variables, use it for feature selection maybe)
8. Use MRI in some way??
9. K-Means Clustering to add a new feature